# Week 12a: Multi-Agent Systems
**Applied Generative AI**
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)
*Developed by Prachi Patel, PhD | Open Curriculum — CC BY 4.0*

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand multi-agent architecture patterns** — Orchestrator-worker, debate, reflection, and supervisor designs
2. **Implement orchestrator-worker systems** — Central coordinator that delegates to specialist agents
3. **Build agent debate and reflection loops** — Agents argue opposing positions or critique their own output for higher quality
4. **Design supervisor architectures with LangGraph** — Hierarchical control with escalation paths
5. **Handle errors and fallbacks in multi-agent systems** — Retry, escalate, circuit breaker, and default response strategies

---
## ⚙️ Setup — Install & Import

In [ ]:
!pip install -q langgraph langchain langchain-openai langchain-core openai transformers

In [ ]:
import os
from getpass import getpass
from typing import TypedDict, Literal

# Set OpenAI API key securely (works in Colab)
if not os.environ.get("OPENAI_API_KEY"):
    try:
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key (or press Enter to skip): ")
    except Exception:
        os.environ["OPENAI_API_KEY"] = ""

HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
print(f"OpenAI API key configured: {HAS_OPENAI}")

# Fallback: use open-source Flan-T5 for students without API keys
if not HAS_OPENAI:
    print("Using Flan-T5 (open-source) as fallback. Some features may be limited.")

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage

if HAS_OPENAI:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
else:
    from transformers import pipeline
    pipe = pipeline("text2text-generation", model="google/flan-t5-base", max_length=256)
    
    class FlanT5Wrapper:
        def invoke(self, messages):
            text = messages[-1].content if hasattr(messages[-1], 'content') else str(messages[-1])
            out = pipe(text, max_new_tokens=150)[0]['generated_text']
            return type('Msg', (), {'content': out})()
    llm = FlanT5Wrapper()

---
## 1. Multi-Agent Architecture Patterns

Multi-agent systems distribute work across specialized agents. Common patterns:

### Orchestrator-Worker
A **central coordinator** delegates tasks to specialist workers (researcher, writer, critic). The orchestrator decides which worker to invoke and in what order.

### Debate
Two or more agents argue **opposing positions** on a question. A judge agent evaluates arguments and synthesizes a conclusion. Improves robustness by surfacing counterarguments.

### Reflection
An agent generates output, then **critiques its own work**, and iterates to improve. Self-reflection loops can converge to higher-quality outputs.

### Supervisor
**Hierarchical control**: a supervisor delegates to team leads, who delegate to workers. Supports escalation when workers fail or need higher-level decisions.

### When to Use Which Pattern

| Pattern | Best For | Pros | Cons |
|---------|----------|------|------|
| **Orchestrator-Worker** | Content pipelines, multi-step workflows | Clear separation of concerns, easy to extend | Single point of failure (orchestrator) |
| **Debate** | High-stakes decisions, reducing bias | Surfaces counterarguments, improves reasoning | Slower, more tokens |
| **Reflection** | Quality-critical outputs (code, reports) | Self-improving, no extra agents | May not converge; risk of over-refinement |
| **Supervisor** | Large teams, complex hierarchies | Escalation, clear accountability | More complex to implement |

---
## 2. Orchestrator-Worker Pattern

We build a content pipeline: **research → write → critique → revise**. The orchestrator delegates to specialist workers based on the current step.

In [ ]:
# State schema for the content pipeline
class ContentState(TypedDict):
    topic: str
    step: str
    research: str
    draft: str
    critique: str
    final: str
    iterations: int

In [ ]:
# Worker agents: researcher, writer, critic
def researcher(state: ContentState) -> dict:
    """Worker: gather key points on the topic."""
    prompt = f"List 3-5 key points about: {state['topic']}. Be concise."
    if HAS_OPENAI:
        resp = llm.invoke([HumanMessage(content=prompt)])
        research = resp.content
    else:
        research = pipe(prompt, max_new_tokens=150)[0]['generated_text']
    return {"research": research, "step": "write"}

def writer(state: ContentState) -> dict:
    """Worker: draft content from research."""
    prompt = f"Topic: {state['topic']}. Key points: {state['research']}. Write a short paragraph (3-4 sentences)."
    if HAS_OPENAI:
        resp = llm.invoke([HumanMessage(content=prompt)])
        draft = resp.content
    else:
        draft = pipe(prompt, max_new_tokens=150)[0]['generated_text']
    return {"draft": draft, "step": "critique"}

def critic(state: ContentState) -> dict:
    """Worker: critique the draft."""
    prompt = f"Critique this draft in 1-2 sentences. Suggest one improvement. Draft: {state['draft']}"
    if HAS_OPENAI:
        resp = llm.invoke([HumanMessage(content=prompt)])
        critique = resp.content
    else:
        critique = pipe(prompt, max_new_tokens=100)[0]['generated_text']
    return {"critique": critique, "step": "revise"}

In [ ]:
def reviser(state: ContentState) -> dict:
    """Worker: improve draft based on critique."""
    prompt = f"Draft: {state['draft']}. Critique: {state['critique']}. Write an improved version (3-4 sentences)."
    if HAS_OPENAI:
        resp = llm.invoke([HumanMessage(content=prompt)])
        final = resp.content
    else:
        final = pipe(prompt, max_new_tokens=150)[0]['generated_text']
    return {"final": final, "step": "done", "iterations": state.get("iterations", 0) + 1}

# Orchestrator: routes to next step
def route(state: ContentState) -> Literal["researcher", "writer", "critic", "reviser", "__end__"]:
    step = state.get("step", "research")
    if step == "research": return "researcher"
    if step == "write": return "writer"
    if step == "critique": return "critic"
    if step == "revise": return "reviser"
    return "__end__"

In [ ]:
# Build the orchestrator-worker graph with LangGraph
workflow = StateGraph(ContentState)
workflow.add_node("researcher", researcher)
workflow.add_node("writer", writer)
workflow.add_node("critic", critic)
workflow.add_node("reviser", reviser)

workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", "critic")
workflow.add_edge("critic", "reviser")
workflow.add_edge("reviser", END)

app = workflow.compile()

# Run full pipeline: research → write → critique → revise
result = app.invoke({
    "topic": "benefits of renewable energy",
    "step": "research",
    "research": "", "draft": "", "critique": "", "final": "", "iterations": 0
})
print("=== Research ==="); print(result["research"])
print("\n=== Draft ==="); print(result["draft"])
print("\n=== Critique ==="); print(result["critique"])
print("\n=== Final ==="); print(result["final"])

In [ ]:
# Optional: visualize the orchestrator-worker graph
from IPython.display import Image
try:
    display(Image(app.get_graph().draw_png()))
except Exception as e:
    print("Graphviz not installed. Run: pip install pygraphviz")

---
## 3. Agent Debate

Two agents argue opposite sides of a question; a judge synthesizes. Debate often improves answer quality by surfacing counterarguments.

In [ ]:
def advocate_pro(state: dict) -> dict:
    """Agent arguing FOR the proposition."""
    q = state["question"]
    prompt = f"Argue FOR this position in 2-3 sentences: {q}"
    if HAS_OPENAI:
        resp = llm.invoke([HumanMessage(content=prompt)])
        arg = resp.content
    else:
        arg = pipe(prompt, max_new_tokens=100)[0]['generated_text']
    return {"pro_argument": arg}

def advocate_con(state: dict) -> dict:
    """Agent arguing AGAINST the proposition."""
    q = state["question"]
    prompt = f"Argue AGAINST this position in 2-3 sentences: {q}"
    if HAS_OPENAI:
        resp = llm.invoke([HumanMessage(content=prompt)])
        arg = resp.content
    else:
        arg = pipe(prompt, max_new_tokens=100)[0]['generated_text']
    return {"con_argument": arg}

def judge(state: dict) -> dict:
    """Judge evaluates both arguments and synthesizes a balanced conclusion."""
    prompt = f"Question: {state['question']}\nPro: {state['pro_argument']}\nCon: {state['con_argument']}\nSynthesize a balanced 2-3 sentence conclusion."
    if HAS_OPENAI:
        resp = llm.invoke([HumanMessage(content=prompt)])
        synthesis = resp.content
    else:
        synthesis = pipe(prompt, max_new_tokens=120)[0]['generated_text']
    return {"synthesis": synthesis}

In [ ]:
class DebateState(TypedDict):
    question: str
    pro_argument: str
    con_argument: str
    synthesis: str

debate_graph = StateGraph(DebateState)
debate_graph.add_node("pro", advocate_pro)
debate_graph.add_node("con", advocate_con)
debate_graph.add_node("judge", judge)

debate_graph.set_entry_point("pro")
debate_graph.add_edge("pro", "con")
debate_graph.add_edge("con", "judge")
debate_graph.add_edge("judge", END)

debate_app = debate_graph.compile()

question = "Should remote work be the default for knowledge workers?"
result = debate_app.invoke({"question": question, "pro_argument": "", "con_argument": "", "synthesis": ""})
print("Question:", question)
print("\nPro:", result["pro_argument"])
print("\nCon:", result["con_argument"])
print("\nSynthesis:", result["synthesis"])

In [ ]:
# Compare: single-agent vs debate
def single_agent_answer(q: str) -> str:
    if HAS_OPENAI:
        resp = llm.invoke([HumanMessage(content=f"Answer in 2-3 sentences: {q}")])
        return resp.content
    return pipe(f"Answer: {q}", max_new_tokens=100)[0]['generated_text']

single = single_agent_answer(question)
debate_synth = result["synthesis"]
print("Single-agent answer:", single)
print("\nDebate synthesis:", debate_synth)
print("\nDebate considers both sides; single-agent may be one-sided.")

---
## 4. Self-Reflection Loop

An agent generates output, critiques it, and improves. We implement a reflection loop with a simple quality check (e.g., length or keyword presence) to show convergence.

In [ ]:
class ReflectionState(TypedDict):
    topic: str
    output: str
    critique: str
    iteration: int
    max_iterations: int
    done: bool

def generate(state: ReflectionState) -> dict:
    """Generate initial or revised output."""
    if state.get("iteration", 0) == 0:
        prompt = f"Write a 2-3 sentence summary about: {state['topic']}"
    else:
        prompt = f"Topic: {state['topic']}. Previous: {state['output']}. Critique: {state['critique']}. Write improved version."
    if HAS_OPENAI:
        resp = llm.invoke([HumanMessage(content=prompt)])
        out = resp.content
    else:
        out = pipe(prompt, max_new_tokens=100)[0]['generated_text']
    return {"output": out}

def critique_output(state: ReflectionState) -> dict:
    """Critique the current output."""
    prompt = f"Critique in 1 sentence. Output: {state['output']}"
    if HAS_OPENAI:
        resp = llm.invoke([HumanMessage(content=prompt)])
        c = resp.content
    else:
        c = pipe(prompt, max_new_tokens=50)[0]['generated_text']
    return {"critique": c, "iteration": state.get("iteration", 0) + 1}

def should_continue(state: ReflectionState) -> Literal["generate", "__end__"]:
    max_it = state.get("max_iterations", 2)
    it = state.get("iteration", 0)
    if it >= max_it: return "__end__"
    return "generate"

In [ ]:
reflection_graph = StateGraph(ReflectionState)
reflection_graph.add_node("generate", generate)
reflection_graph.add_node("critique", critique_output)

reflection_graph.set_entry_point("generate")
reflection_graph.add_edge("generate", "critique")
reflection_graph.add_conditional_edges("critique", should_continue, {"generate": "generate", "__end__": END})

reflection_app = reflection_graph.compile()

# Simple quality metric: output length (proxy for completeness)
def quality_score(text: str) -> float:
    return min(1.0, len(text.split()) / 30)  # 30 words = "good enough"

topic = "machine learning in healthcare"
r = reflection_app.invoke({
    "topic": topic, "output": "", "critique": "", "iteration": 0,
    "max_iterations": 2, "done": False
})
print("Final output:", r["output"])
print("Quality score (word-based proxy):", round(quality_score(r["output"]), 2))
print("Iterations:", r["iteration"])

---
## 5. Supervisor Architecture

Hierarchical structure: **supervisor → team leads → workers**. When a worker fails, the supervisor can redirect or escalate.

### Nested Graphs (Advanced)

LangGraph supports **nested subgraphs**: a node can be another compiled graph. For the supervisor, we could implement each level (worker, team_lead, supervisor) as its own subgraph with internal steps. Here we use a flat graph for clarity; the same routing logic applies.

In [ ]:
class SupervisorState(TypedDict):
    task: str
    result: str
    status: str  # ok, failed, escalate
    level: str   # worker, lead, supervisor

def worker(state: SupervisorState) -> dict:
    """Low-level worker; may fail on complex tasks."""
    task = state["task"]
    # Simulate failure for tasks containing "complex"
    if "complex" in task.lower():
        return {"result": "", "status": "failed", "level": "worker"}
    if HAS_OPENAI:
        resp = llm.invoke([HumanMessage(content=f"Briefly answer: {task}")])
        out = resp.content
    else:
        out = pipe(task, max_new_tokens=80)[0]['generated_text']
    return {"result": out, "status": "ok", "level": "worker"}

def team_lead(state: SupervisorState) -> dict:
    """Team lead handles escalated tasks; fails on 'very complex' to escalate to supervisor."""
    task = state["task"]
    if "very complex" in task.lower():
        return {"result": "", "status": "failed", "level": "lead"}
    if HAS_OPENAI:
        resp = llm.invoke([HumanMessage(content=f"As a team lead, provide a structured answer: {task}")])
        out = resp.content
    else:
        out = pipe(f"Answer: {task}", max_new_tokens=100)[0]['generated_text']
    return {"result": out, "status": "ok", "level": "lead"}

def supervisor_node(state: SupervisorState) -> dict:
    """Supervisor makes final decision on hard cases."""
    task = state["task"]
    if HAS_OPENAI:
        resp = llm.invoke([HumanMessage(content=f"As supervisor, give authoritative answer: {task}")])
        out = resp.content
    else:
        out = pipe(f"Final answer: {task}", max_new_tokens=120)[0]['generated_text']
    return {"result": out, "status": "ok", "level": "supervisor"}

In [ ]:
def route_supervisor(state: SupervisorState) -> Literal["worker", "team_lead", "supervisor", "__end__"]:
    if state.get("result") and state.get("status") == "ok":
        return "__end__"
    level = state.get("level", "worker")
    status = state.get("status", "")
    if level == "worker" and status == "failed":
        return "team_lead"
    if level == "lead" and status == "failed":
        return "supervisor"
    if level == "supervisor":
        return "__end__"
    return "worker"

sup_graph = StateGraph(SupervisorState)
sup_graph.add_node("worker", worker)
sup_graph.add_node("team_lead", team_lead)
sup_graph.add_node("supervisor", supervisor_node)

sup_graph.set_entry_point("worker")
sup_graph.add_conditional_edges("worker", lambda s: "team_lead" if s.get("status") == "failed" else "__end__",
                                 {"team_lead": "team_lead", "__end__": END})
sup_graph.add_conditional_edges("team_lead", lambda s: "supervisor" if s.get("status") == "failed" else "__end__",
                                 {"supervisor": "supervisor", "__end__": END})
sup_graph.add_edge("supervisor", END)

sup_app = sup_graph.compile()

# Test: simple task (worker succeeds)
r1 = sup_app.invoke({"task": "What is 2+2?", "result": "", "status": "", "level": ""})
print("Simple task:", r1["result"][:80], "... | Level:", r1["level"])

# Test: complex task (escalate to lead)
r2 = sup_app.invoke({"task": "Explain this complex quantum computing concept", "result": "", "status": "", "level": ""})
print("\nComplex task (escalated):", r2["result"][:80], "... | Level:", r2["level"])

# Test: very complex task (escalate worker -> lead -> supervisor)
r3 = sup_app.invoke({"task": "Explain this very complex quantum entanglement concept", "result": "", "status": "", "level": ""})
print("\nVery complex task (full escalation):", r3["result"][:80], "... | Level:", r3["level"])

---
## 6. Error Handling and Fallbacks

Multi-agent systems can fail in many ways: **timeout**, **bad output**, **infinite loop**. We implement fallback strategies: retry, escalate, default response, and a simple circuit breaker.

In [ ]:
import time

def agent_with_retry(prompt: str, max_retries: int = 2) -> str:
    """Retry failed calls up to max_retries times."""
    for attempt in range(max_retries + 1):
        try:
            if HAS_OPENAI:
                resp = llm.invoke([HumanMessage(content=prompt)])
                return resp.content
            return pipe(prompt, max_new_tokens=80)[0]['generated_text']
        except Exception as e:
            if attempt == max_retries:
                return f"[Fallback] Error after {max_retries} retries: {str(e)[:50]}"
            time.sleep(0.5)
    return "[Fallback] Unknown error"

def agent_with_timeout(prompt: str, timeout_sec: float = 5.0) -> str:
    """Timeout wrapper; return default on timeout. Uses threading (cross-platform)."""
    import threading
    result = [None]
    def run():
        result[0] = agent_with_retry(prompt)
    t = threading.Thread(target=run)
    t.start()
    t.join(timeout=timeout_sec)
    if t.is_alive():
        return "[Fallback] Request timed out"
    return result[0] or "[Fallback] Default response"

print("Retry example:", agent_with_retry("Say hello in one word")[:60])

In [ ]:
class CircuitBreaker:
    """Simple circuit breaker: after N failures, stop calling and return fallback."""
    def __init__(self, failure_threshold: int = 3):
        self.failures = 0
        self.threshold = failure_threshold
        self.open = False

    def call(self, fn, *args, fallback="[Circuit open] Service temporarily unavailable", **kwargs):
        if self.open:
            return fallback
        try:
            out = fn(*args, **kwargs)
            self.failures = 0
            return out
        except Exception:
            self.failures += 1
            if self.failures >= self.threshold:
                self.open = True
            raise

cb = CircuitBreaker(failure_threshold=2)
def flaky_agent(x):
    if "fail" in x.lower(): raise RuntimeError("Simulated failure")
    return "OK"

print(cb.call(flaky_agent, "hello"))
try: cb.call(flaky_agent, "fail")
except: pass
try: cb.call(flaky_agent, "fail again")
except: pass
print("After 2 failures:", cb.call(flaky_agent, "hello"))  # Circuit open

---
## 7. Exercises

1. **Exercise 1: Content Creation Pipeline** — Build an orchestrator-worker system for: research → draft → edit → publish. Add an "editor" worker that checks grammar and style before a "publisher" outputs the final content.

2. **Exercise 2: AI Hiring Debate** — Implement a debate between two agents on *"Should AI systems be allowed to make hiring decisions?"* Use a judge to synthesize. Evaluate: Does the synthesis fairly represent both sides?

3. **Exercise 3: Error Handling** — Add retry logic, timeout, or circuit breaker to any multi-agent system from this notebook. Test with simulated failures.

In [ ]:
# Exercise 1: Content Creation Pipeline — add editor and publisher workers
# Exercise 2: AI Hiring Debate — run debate_app.invoke({"question": "Should AI systems be allowed to make hiring decisions?", ...})
# Exercise 3: Wrap any agent above with agent_with_retry or CircuitBreaker

---
## 8. Responsible AI Reflection

Multi-agent systems can produce impressive results, but they also **diffuse responsibility**. When three agents collaborate to produce an output, who is accountable if that output causes harm? The orchestrator? The worker that generated the problematic content? The designer?

As you build multi-agent systems, consider:

> **How do you maintain accountability in systems where no single component is responsible for the final output?**

Possible approaches: audit trails (which agent produced what), human-in-the-loop checkpoints, clear ownership of the orchestrator, and transparent documentation of agent roles.

---
## Summary & Next Steps

**Summary:**
- **Orchestrator-Worker**: Central coordinator delegates to specialists (research, write, critique, revise)
- **Debate**: Pro/con agents + judge for balanced synthesis
- **Reflection**: Self-critique loop for iterative improvement
- **Supervisor**: Hierarchical escalation when workers fail
- **Error handling**: Retry, timeout, circuit breaker, fallback responses

**Next:**
- **Week 12b**: Advanced multi-agent topics (swarm intelligence, emergent behavior)
- **Week 12c**: Production deployment of multi-agent systems